In [ ]:
import os

import pandas as pd
import numpy as np
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt

from matplotlib.transforms import ScaledTranslation
from matplotlib.lines import Line2D

import loading as ld
import processing as prc
import visualization as viz

path_to_images = '/Users/Guille/Desktop/india_power/images'
path_to_output = '/Users/Guille/Desktop/india_power/output_csvs'
path_to_csvs = '/Users/Guille/Desktop/india_power/gridpath_india_viz/csvs'
path_to_input = '/Users/Guille/Desktop/india_power/input_data/'

In [ ]:
tech_labels_ = pd.read_csv(path_to_csvs + '/technology_labels.csv') 
load_zones_ = pd.read_csv(path_to_input + r"/load_zones.csv").drop(columns = ['state', 'rps_zone'])

In [ ]:
# Grab data from databases for plotting new and existing capacity
def _load_capacity_retirements(scen_labels_):

    # Load project capacity table and process them from database
    def __load_pipeline_and_existing_csv(df_, scenario):

        df_ = df_[['project', 
                   'period', 
                   'technology', 
                   'load_zone', 
                   'capacity_mw', 
                   'energy_capacity_mwh']]

        df_  = df_.loc[~df_['project'].isin(['_new'])].reset_index(drop = True)
        
        df_['status'] = 'existing'
        
        df_.loc[df_['project'].str.contains('_pipeline'), 'status'] = 'pipeline'
        df_.loc[df_['project'].str.contains('_new'), 'status'] = 'new'
        df_.loc[df_['project'].str.contains('2029'), 'status'] = 'pipeline'

        df_ = df_.groupby(['period', 
                           'technology', 
                           'load_zone',
                           'status']).agg({'capacity_mw': 'sum', 
                                           'energy_capacity_mwh': 'sum'})

        df_ = df_.reset_index(drop = False).rename(columns = {'energy_capacity_mwh': 'capacity_mwh'})

        df_['scenario'] = scenario

        return df_
    
    dfs_ = []
    # Open connection: open database and grab meta-data
    for scen, path in zip(scen_labels_['scenario'],  scen_labels_['path']):
        print(scen, path)
        dir_name  = r'{}/{}'.format(path, scen)
        capacity_ = pd.read_csv(dir_name + r'/results/project_period.csv', low_memory = False)
        df_       = __load_pipeline_and_existing_csv(capacity_, scen)
        
        # Load specified capacity from csv files
        dfs_.append(df_)

    return pd.concat(dfs_, axis = 0).reset_index(drop = True)

# Grab data from databases for plotting new and existing capacity
def _load_new_build_capacity(scen_labels_):

    # Load project capacity table and process them from database
    def __load_pipeline_and_existing_csv(df_, scenario):

        df_ = df_[['project', 
                   'period', 
                   'technology', 
                   'load_zone', 
                   'capacity_mw',
                   'energy_capacity_mwh',
                   'new_build_mw', 
                   'new_build_mwh']]
        
        df_p_ = df_.copy()

        df_p_['status']                                                 = 'existing'
        df_p_.loc[df_p_['project'].str.contains('_pipeline'), 'status'] = 'pipeline'
        df_p_.loc[df_p_['project'].str.contains('_new'), 'status']      = 'new'
        df_p_.loc[df_p_['project'].str.contains('2029'), 'status']      = 'pipeline'
        
        idx_ = (df_p_['status'] == 'pipeline') | (df_p_['status'] == 'existing')
        for project in df_p_.loc[idx_, 'project']:
            
            df_p_.loc[df_p_['project'] == project, 'new_build_mw']  = df_p_.loc[df_p_['project'] == project, 'capacity_mw'].diff().fillna(0) 
            df_p_.loc[df_p_['project'] == project, 'new_build_mwh'] = df_p_.loc[df_p_['project'] == project, 'energy_capacity_mwh'].diff().fillna(0) 
        
        df_p_.loc[df_p_['status'] == 'pipeline', 'status'] = 'new'
        
        df_p_ = df_p_.groupby(['period', 
                               'technology', 
                               'load_zone',
                               'status']).agg({'new_build_mw': 'sum', 
                                               'new_build_mwh': 'sum'}).reset_index(drop = False)

        
        df_p_ = df_p_[['period', 
                       'technology', 
                       'load_zone', 
                       'status',
                       'new_build_mw', 
                       'new_build_mwh']]
        
        df_p_ = df_p_.rename(columns = {'new_build_mw': 'capacity_mw',
                                        'new_build_mwh': 'capacity_mwh'})

        df_p_['scenario'] = scenario

        return df_p_
    
    dfs_ = []
    # Open connection: open database and grab meta-data
    for scen, path in zip(scen_labels_['scenario'],  scen_labels_['path']):
        print(scen, path)
        dir_name  = r'{}/{}'.format(path, scen)
        capacity_ = pd.read_csv(dir_name + r'/results/project_period.csv', low_memory = False)
        df_       = __load_pipeline_and_existing_csv(capacity_, scen)
        
        # Load specified capacity from CSV files
        dfs_.append(df_)

    return pd.concat(dfs_, axis = 0).reset_index(drop = True)

In [ ]:
def _group_capacity_technologies(capacity_, tech_labels_):

    for group in tech_labels_['group'].unique():
        capacity_.loc[capacity_['technology'].isin(tech_labels_.loc[tech_labels_['group'] == group, 'technology']), 'technology'] = group
        
    capacity_ = capacity_.groupby(['period', 
                                   'technology', 
                                   'load_zone', 
                                   'status', 
                                   'scenario']).agg({'capacity_mw': 'sum', 
                                                     'capacity_mwh': 'sum'})

    return capacity_.reset_index(drop = False)


def _group_by_zone(df_, scen_labels_, columns_ = ['period', 
                                                  'technology', 
                                                  'scenario', 
                                                  'load_zone', 
                                                  'status']):
    
    dfs_ = []
    # Open connection: open database and grab meta-data
    for scen, zone in zip(scen_labels_['scenario'], scen_labels_['zone']):
        
        df_p_ = df_.loc[df_['scenario'] == scen].copy()
        
        idx_ = df_p_['load_zone'] == zone
        
        if idx_.sum() == 0.:
            df_p_['load_zone'] = 'all'
        else:
            df_p_ = df_p_.loc[idx_].reset_index(drop = True)
        
        dfs_.append(df_p_.groupby(columns_).sum().reset_index(drop = False))
        
    df_                 = pd.concat(dfs_, axis = 0).reset_index(drop = True)
    df_['period']       = df_['period'].astype(int)
    df_['capacity_mw']  = df_['capacity_mw'].astype(float)
    df_['capacity_mwh'] = df_['capacity_mwh'].astype(float)

    return df_

In [ ]:
def _plot_retirements_and_pipeline_capacity(ax, df_, scens_label_, tech_label_, 
                                            units        = 1e3,
                                            ylabel       = 'Retirements & Pipeline (GW)',
                                            y_lim_max = 500.,
                                            y_grid_inc   = 100,
                                            legend       = True,
                                            title        = ''):

    def __make_new_and_existing_capacit_legend(df_, techs_, colors_, ax):

        ax.bar(0., 0., 0., bottom    = 0.,
                           label     = 'Existing',
                           color     = 'None',
                           lw        = 0.,
                           hatch     = 'xx',
                           edgecolor = 'lightgray',
                           zorder    = 10)

        # for tech, i_tech in zip(techs_, range(len(techs_))):
        #     idx_ = df_['technology'] == tech
        #     if idx_.sum() > 0:
        #         if df_.loc[idx_, 'capacity_mw'].sum() != 0:
        #             ax.bar(0., 0., 0., bottom = 0.,
        #                                color  = colors_[i_tech],
        #                                label  = tech,
        #                                zorder = 10)

    periods_     = np.sort(df_['period'].unique())
    scens_       = scens_label_['scenario'].unique()    
    techs_, idx_ = np.unique(tech_label_['group'], return_index = True)
    colors_      = tech_label_.loc[tech_label_.index[idx_], 'group_color'].to_list()

    ticks_    = []
    labels_   = []
    offsets_  = []
    lengths_  = []
    x_period_ = []
    
    i_scen = 0
    offset = 0
    y_max  = 0
    
    width  = 0.75
    x_     = np.linspace(0, len(periods_) - 1, len(periods_))
    
    for scen, label, zone in zip(scens_label_['scenario'], scens_label_['label'], scens_label_['zone']):
        print(scen, zone)
        df_p_ = df_.copy()

        idx_ = (df_p_['scenario'] == scen) & (df_p_['load_zone'] == zone)
                
        for period, i_period in zip(periods_, range(len(periods_))):
            idx_1_ = idx_ & (df_p_['period'] == period)
            if (i_scen == 0) & (i_period == 0): 
                __make_new_and_existing_capacit_legend(df_p_, techs_, colors_, ax)

            for tech, i_tech in zip(techs_, range(len(techs_))):
                idx_2_ = idx_1_ & (df_p_['status'] == 'existing') & (df_p_['technology'] == tech)

                if idx_2_.sum() == 1.:
                    power = df_p_.loc[idx_2_, 'capacity_mw'].to_numpy()[0]
                    color = tech_label_.loc[tech_label_['group'] == tech, 'group_color'].unique()

                    ax.bar(x_[i_period], power/units, width, bottom    = offset/units,
                                                             color     = color,
                                                             lw        = 0.,
                                                             hatch     = 'x',
                                                             edgecolor = 'lightgray', zorder = 10)

                    offset += power

            for tech, i_tech in zip(techs_, range(len(techs_))):
                idx_3_ = idx_1_ & (df_p_['technology'] == tech) & (df_p_['status'] == 'pipeline')
                if idx_3_.sum() == 1.:
                    power = df_p_.loc[idx_3_, 'capacity_mw'].to_numpy()[0]
                    color = tech_label_.loc[tech_label_['group'] == tech, 'group_color'].unique()

                    ax.bar(x_[i_period], power/units, width, bottom = offset/units,
                                                             color  = color,
                                                             zorder = 10)

                    offset += power

            if y_max < offset:
                y_max = offset

            ticks_.append(x_[i_period])
            labels_.append(label)
            offsets_.append(offset/units)
            x_period_.append(x_[i_period])

            offset = 0.

        x_ = x_ + .9/len(scens_)
        i_scen += 1
        
    z_ = x_ - .9/len(scens_)

    x_period_ = np.mean(np.array(x_period_).reshape(len(scens_), len(periods_)), axis = 0)
    y_period_ = np.max(np.array(offsets_).reshape(len(scens_), len(periods_)), axis = 0)

    ax.axhline(0., linewidth = .75,
                   linestyle = '-',
                   color     = 'k',
                   clip_on   = False,
                   zorder    = 10)

    ax.set_ylabel(ylabel, fontsize = 18)

    if legend:
        ax.legend(loc            = 'upper left',
                  frameon        = False,
                  prop           = {'size': 16})

    if y_lim_max != None:
        ax.set_ylim(0., y_lim_max)
        y_max = y_lim_max
    else:
        ax.set_ylim(0., y_period_.max()*1.2)
        y_max /= units
        
    N_steps  = int(np.ceil(y_max/y_grid_inc))
    y_ticks_ = np.linspace(0, int(N_steps*y_grid_inc), N_steps + 1, dtype = int)
    
    ax.set_yticks(y_ticks_, y_ticks_)
    ax.set_xticks(x_period_, periods_)

    ax.xaxis.set_tick_params(labelsize = 19, 
                             left      = False)
    
    ax.yaxis.set_tick_params(labelsize = 15, 
                             left      = False)
    
    ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))

    ax.axhline(0, linewidth = .5, 
                  linestyle = '-', 
                  color     = 'k', 
                  clip_on   = False, 
                  zorder    = 10)
    
    ax.set_title(title, fontsize = 20, y = 0.912)

    ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax.grid(axis = 'y')

# Capacity Retirements

In [ ]:
cost_ = {}
cost_['scen_labels'] = pd.read_csv(path_to_csvs + '/retirements-scenario_labels.csv')

retirements_ = _load_capacity_retirements(cost_['scen_labels'])
grouped_retirements_ = _group_capacity_technologies(retirements_, tech_labels_)
grouped_retirements_ = _group_by_zone(grouped_retirements_, cost_['scen_labels'])

In [ ]:
fig = plt.figure(figsize = (7, 5.5))
_ax = plt.subplot(111)

_plot_retirements_and_pipeline_capacity(_ax, grouped_retirements_, cost_['scen_labels'] , tech_labels_,
                                        ylabel = 'Retirements & Pipeline Capacity (GW)') 

plt.savefig(path_to_images + r'/group_retirements_mw.pdf', bbox_inches = 'tight', 
                                                           dpi         = 300)
plt.show()

In [ ]:
retirements_ = _load_capacity_retirements(cost_['scen_labels'])
grouped_retirements_ = _group_capacity_technologies(retirements_, tech_labels_)
print(grouped_retirements_['scenario'].unique())

grouped_retirements_ = grouped_retirements_.loc[grouped_retirements_['scenario'] == 'VREmid_STmid_CONVmid_H2_RES_8PRM_CC_50RPS_90CAP_500GW_PIERmid']
grouped_retirements_ = grouped_retirements_.loc[~(grouped_retirements_['status'] == 'new')]
grouped_retirements_ = grouped_retirements_.loc[grouped_retirements_['technology'] == 'Coal']

grouped_retirements_ = grouped_retirements_.groupby(['period', 
                                                     'technology',
                                                     'status']).agg({'capacity_mw': 'sum'}).reset_index(drop = False)
print(grouped_retirements_)

# New Build Capacity

In [ ]:
new_build_ = _load_new_build_capacity(scen_labels_)
grouped_new_build_ = _group_capacity_technologies(new_build_, tech_labels_)
grouped_new_build_ = _group_by_zone(grouped_new_build_, scen_labels_)

In [ ]:
fig = plt.figure(figsize = (10, 7.5))
_ax = plt.subplot(111)

_plot_new_build_capacity(_ax, grouped_new_build_, scen_labels_, tech_labels_,
                         ylabel = 'New Build Capacity (GW)') 

plt.savefig(path_to_images + r'/group_new_build_mw.png', bbox_inches = 'tight', 
                                                         dpi         = 300)
plt.show()

In [ ]:
# def _plot_retirements_and_pipeline_capacity(ax, df_, tech_label_, 
#                                             units        = 1e3,
#                                             ylabel       = 'Retirements & Pipeline (GW)',
#                                             y_lim_max    = 325.,
#                                             y_grid_inc   = 100,
#                                             legend       = True,
#                                             title        = ''):

#     def __make_new_and_existing_capacit_legend(df_, techs_, colors_, ax):

#         ax.bar(0., 0., 0., bottom    = 0.,
#                            label     = 'Existing',
#                            color     = 'None',
#                            lw        = 0.,
#                            hatch     = 'xx',
#                            edgecolor = 'lightgray',
#                            zorder    = 10)
#         for tech, i_tech in zip(techs_, range(len(techs_))):
#             idx_ = df_['technology'] == tech
#             if idx_.sum() > 0:
#                 if df_.loc[idx_, 'capacity_mw'].sum() != 0:
#                     ax.bar(0., 0., 0., bottom = 0.,
#                                        color  = colors_[i_tech],
#                                        label  = tech,
#                                        zorder = 10)
        
#     periods_     = np.sort(df_['period'].unique())
#     techs_, idx_ = np.unique(tech_label_['group'], return_index = True)
#     colors_      = tech_label_.loc[tech_label_.index[idx_], 'group_color'].to_list()

#     print(df_)
#     ticks_    = []
#     labels_   = []
#     offsets_  = []
#     lengths_  = []
#     x_period_ = []
    
#     i_scen = 0
#     offset = 0
#     y_max  = 0
    
#     width  = 0.75
#     x_     = np.linspace(0, len(periods_) - 1, len(periods_))
    
           
#     for period, i_period in zip(periods_, range(len(periods_))):

#         if i_period == 0: 
#             __make_new_and_existing_capacit_legend(df_, techs_, colors_, ax)


#         for tech, i_tech in zip(techs_, range(len(techs_))):
#             idx_ = (df_['period'] == period) & (df_['status'] == 'existing') & (df_['technology'] == tech)

#             if idx_.sum() == 1.:
#                 power = df_.loc[idx_, 'capacity_mw'].to_numpy()[0]
#                 color = tech_label_.loc[tech_label_['group'] == tech, 'group_color'].unique()

#                 ax.bar(x_[i_period], power/units, width, 
#                        bottom    = offset/units,
#                        color     = color,
#                        lw        = 0.,
#                        hatch     = 'x',
#                        edgecolor = 'lightgray', 
#                        zorder    = 10)
                
#                 print(offset/units, power/units)
#                 offset += power

#         for tech, i_tech in zip(techs_, range(len(techs_))):
#             idx_ = (df_['period'] == period) & (df_['technology'] == tech) & (df_['status'] == 'pipeline')
#             if idx_.sum() == 1.:
#                 power = df_.loc[idx_, 'capacity_mw'].to_numpy()[0]
#                 color = tech_label_.loc[tech_label_['group'] == tech, 'group_color'].unique()
#                 print(offset/units, power/units)

#                 ax.bar(x_[i_period], power/units, width, 
#                        bottom = offset/units,
#                        color  = color,
#                        zorder = 10)

#                 offset += power

#             if y_max < offset:
#                 y_max = offset

#         ticks_.append(x_[i_period])
#         labels_.append(label)
#         offsets_.append(offset/units)
#         x_period_.append(x_[i_period])

#         offset = 0.

        
#     ax.axhline(0., linewidth = .75,
#                    linestyle = '-',
#                    color     = 'k',
#                    clip_on   = False,
#                    zorder    = 10)

#     ax.set_ylabel(ylabel, fontsize = 18)

#     if legend:
#         ax.legend(loc            = 'upper right',
#                   frameon        = False,
#                   prop           = {'size': 16})

#     if y_lim_max != None:
#         ax.set_ylim(0., y_lim_max)
#         y_max = y_lim_max
#     else:
#         ax.set_ylim(0., y_period_.max()*1.2)
#         y_max /= units
        
#     # N_steps  = int(np.ceil(y_max/y_grid_inc))
#     # y_ticks_ = np.linspace(0, int(N_steps*y_grid_inc), N_steps + 1, dtype = int)
    
#     # ax.set_yticks(y_ticks_, y_ticks_)
#     ax.set_xticks(x_period_, periods_)

#     ax.xaxis.set_tick_params(labelsize = 19, 
#                              left      = False)
    
#     ax.yaxis.set_tick_params(labelsize = 15, 
#                              left      = False)
    
#     ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))

#     ax.axhline(0, linewidth = .5, 
#                   linestyle = '-', 
#                   color     = 'k', 
#                   clip_on   = False, 
#                   zorder    = 10)
    
#     ax.set_title(title, fontsize = 20, y = 0.912)

#     ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
#     ax.grid(axis = 'y')

# fig = plt.figure(figsize = (10, 6.25))
# _ax = plt.subplot(111)

# _plot_retirements_and_pipeline_capacity(_ax, grouped_retirements_, tech_labels_,
#                                         ylabel = 'Coal Retirements & Pipeline Capacity (GW)') 

# plt.savefig(path_to_images + r'/coal_retirements_mw.pdf', 
#             bbox_inches = 'tight', 
#             dpi         = 300)
# plt.show()